# 06 Exposure Analysis

This notebook estimates who and what is at risk inside high-risk flood zones.

## What you will do

1. Convert High and Very High flood classes to polygons.
2. Intersect flood zones with buildings.
3. Intersect flood zones with roads and calculate affected length.
4. Sum exposed population from a population raster.
5. Review the exposure summary table.

## Step 1: Load configuration and check required flood-zone input

In [ ]:
from pathlib import Path
import pandas as pd
from floodsense.config import load_config

config = load_config()
paths = config['paths']
classified_raster = Path(paths['processed_rasters']) / 'susceptibility_class.tif'
print('Classified raster:', classified_raster)
print('Exists:', classified_raster.exists())

## Step 2: Check exposure datasets

The workflow looks for cleaned/preprocessed exposure layers:

- Buildings: `data/interim/cleaned/buildings.gpkg`
- Roads: `data/interim/cleaned/roads.gpkg`
- Population: `data/interim/reprojected/population.tif`

Missing exposure datasets are skipped, and the summary uses only available indicators.

In [ ]:
expected_inputs = {
    'buildings': Path(paths['interim_cleaned']) / 'buildings.gpkg',
    'roads': Path(paths['interim_cleaned']) / 'roads.gpkg',
    'population': Path(paths['interim_reprojected']) / 'population.tif',
}
for name, path in expected_inputs.items():
    print(f'{name:12s}: {path} | exists={path.exists()}')

## Step 3: Run exposure workflow

Outputs:

- `data/processed/vectors/high_risk_flood_zones.gpkg`
- `data/processed/vectors/exposed_buildings.gpkg`
- `data/processed/vectors/exposed_roads.gpkg`
- `data/processed/tables/exposure_summary.csv`
- `outputs/maps/exposure_preview.png`

In [ ]:
from floodsense.exposure import run_exposure_workflow

if classified_raster.exists():
    run_exposure_workflow(config)
else:
    print('Skipped exposure analysis because susceptibility_class.tif is missing.')

## Step 4: Review exposure summary

In [ ]:
summary_path = Path(paths['processed_tables']) / 'exposure_summary.csv'
if summary_path.exists():
    exposure_summary = pd.read_csv(summary_path)
    display(exposure_summary)
else:
    print('Exposure summary not available yet.')

## Step 5: Inspect exposure vector outputs

In [ ]:
from floodsense.paths import list_input_files

for file in list_input_files(paths['processed_vectors'], ['.gpkg', '.geojson']):
    if any(token in file.name for token in ['flood_zones', 'exposed']):
        print(file.name)

## What to check before moving on

- Are exposed roads/buildings spatially reasonable?
- Does exposed population require aggregation by ward/community?
- Are key infrastructure datasets available for a stronger portfolio version?

Next notebook: `07_priority_index_decision_support.ipynb`.